In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import StackingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import gmdh  # библиотека МГУА

# 1️⃣ Загрузка данных
print("📊 Загрузка данных...")
data = fetch_california_housing()
X, y = data.data, data.target
feature_names = data.feature_names

# 2️⃣ Предобработка
# (в этом датасете пропусков нет)

# 3️⃣ Разделение
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Масштабирование (для MLP и МГУА)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4️⃣ Обучение моделей
print("\n🔧 Обучение моделей...")
results = {}

# 🧱 Стекинг (регрессия)
print("  • StackingRegressor...")
stacking_reg = StackingRegressor(
    estimators=[
        ('rf', RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)),
        ('gb', GradientBoostingRegressor(n_estimators=50, random_state=42)),
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=3,
    n_jobs=-1
)
t0 = time.time()
stacking_reg.fit(X_train_scaled, y_train)
t_stack = time.time() - t0
y_pred_stack = stacking_reg.predict(X_test_scaled)
results['Stacking'] = {'time': t_stack, 'y_pred': y_pred_stack}

# 🧠 MLP (регрессия)
print("  • MLPRegressor...")
mlp_reg = MLPRegressor(
    hidden_layer_sizes=(100, 50),
    activation='relu',
    solver='adam',
    alpha=0.001,
    max_iter=300,
    random_state=42,
    early_stopping=True
)
t0 = time.time()
mlp_reg.fit(X_train_scaled, y_train)
t_mlp = time.time() - t0
y_pred_mlp = mlp_reg.predict(X_test_scaled)
results['MLP'] = {'time': t_mlp, 'y_pred': y_pred_mlp}

# 🧮 Импортируем gmdh правильно
try:
    import gmdh
    print("  ✅ GMDH библиотека импортирована успешно")
    gmdh_available = True
except ImportError:
    print("  ⚠️ Библиотека gmdh не найдена. Установите: pip install gmdh")
    gmdh_available = False
    results['GMDH_COMBI'] = None
    results['GMDH_MULTI'] = None

# 🧮 МГУА: линейный метод COMBI
if gmdh_available:
    print("  • GMDH COMBI (linear)...")
    try:
        combi_model = gmdh.Combi()
        t0 = time.time()
        combi_model.fit(X_train_scaled, y_train,
                       verbose=0,  # поставь 1 для вывода прогресса
                       n_jobs=-1,
                       test_size=0.2,
                       limit=0,
                       criterion=gmdh.Criterion(gmdh.CriterionType.REGULARITY))
        t_combi = time.time() - t0
        y_pred_combi = combi_model.predict(X_test_scaled)
        results['GMDH_COMBI'] = {'time': t_combi, 'y_pred': y_pred_combi}
        print(f"    ✅ Обучено за {t_combi:.2f} сек")
    except Exception as e:
        print(f"  ❌ Ошибка GMDH COMBI: {e}")
        results['GMDH_COMBI'] = None

    # 🧮 МГУА: нелинейный метод MULTI
    print("  • GMDH MULTI (nonlinear)...")
    try:
        multi_model = gmdh.Multi()
        t0 = time.time()
        multi_model.fit(X_train_scaled, y_train,
                       verbose=0,
                       n_jobs=-1,
                       test_size=0.2,
                       k_best=5,
                       limit=0,
                       criterion=gmdh.Criterion(gmdh.CriterionType.REGULARITY))
        t_multi = time.time() - t0
        y_pred_multi = multi_model.predict(X_test_scaled)
        results['GMDH_MULTI'] = {'time': t_multi, 'y_pred': y_pred_multi}
        print(f"    ✅ Обучено за {t_multi:.2f} сек")
    except Exception as e:
        print(f"  ❌ Ошибка GMDH MULTI: {e}")
        results['GMDH_MULTI'] = None

# 5️⃣ Оценка качества (метрики для регрессии)
print("\n📈 СРАВНЕНИЕ МОДЕЛЕЙ НА ТЕСТЕ:")
print(f"{'Модель':<20} | {'RMSE':>10} | {'MAE':>10} | {'R²':>10} | {'Время':>8}")
print("-" * 70)
for name, res in results.items():
    if res is None:
        print(f"{name:<20} | {'-':>10} | {'-':>10} | {'-':>10} | {'-':>8}")
        continue
    y_pred = res['y_pred']
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    t = res['time']
    print(f"{name:<20} | {rmse:>10.4f} | {mae:>10.4f} | {r2:>10.4f} | {t:>8.2f}")

# 6️⃣ Важность признаков (на примере Random Forest как референса)
from sklearn.ensemble import RandomForestRegressor
rf_ref = RandomForestRegressor(n_estimators=100, random_state=42)
rf_ref.fit(X_train, y_train)
importances = rf_ref.feature_importances_
indices = np.argsort(importances)[::-1][:10]

📊 Загрузка данных...

🔧 Обучение моделей...
  • StackingRegressor...
  • MLPRegressor...
  ✅ GMDH библиотека импортирована успешно
  • GMDH COMBI (linear)...
    ✅ Обучено за 0.13 сек
  • GMDH MULTI (nonlinear)...
    ✅ Обучено за 0.07 сек

📈 СРАВНЕНИЕ МОДЕЛЕЙ НА ТЕСТЕ:
Модель               |       RMSE |        MAE |         R² |    Время
----------------------------------------------------------------------
Stacking             |     0.5089 |     0.3326 |     0.8043 |    50.59
MLP                  |     0.5293 |     0.3585 |     0.7883 |    26.85
GMDH_COMBI           |     0.7375 |     0.5303 |     0.5889 |     0.13
GMDH_MULTI           |     0.7375 |     0.5303 |     0.5889 |     0.07
